# 02 · Building the contexts ("haystacks")

Each context is a stack of serialized orders, one per line, at a fixed size in **Claude tokens**: 32K, 128K, 512K and 2M.
Needs `ANTHROPIC_API_KEY` in `.env` (token counting is free). Labeling happens *after* this notebook,
and only for the reviews that end up inside a context.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
from rlm_logistics.data import build_contexts, build_orders, claude_token_counter, load_tables, serialize

load_dotenv("../.env")
ROOT = Path("..")
SIZES = {"32k": 32_000, "128k": 128_000, "512k": 512_000, "2m": 2_000_000}
MODEL = yaml.safe_load(open(ROOT / "configs/models.yaml"))["labeler"]["model"]

## 1. Records: commented orders in a fixed random order
We shuffle once with seed 42. Every context is a **prefix** of this list, so the contexts are nested (32K ⊂ 128K ⊂ 512K ⊂ 2M).
Nesting means that when a method fails at 512K but not at 128K, the only thing that changed is the amount of text.

In [ ]:
records = (build_orders(load_tables()).dropna(subset=["comment"])
           .sample(frac=1, random_state=42).reset_index(drop=True))
lines = records.apply(serialize, axis=1).tolist()
print(f"{len(records):,} commented orders\n\n{lines[0]}")

## 2. Counting tokens the way Claude does
Different models split text into tokens differently, so we use Claude's own `count_tokens` endpoint (free).
Portuguese text and long hex order ids cost more tokens than the "4 characters per token" rule of thumb suggests.

In [ ]:
count = claude_token_counter(MODEL)
sample = "\n".join(lines[:200])
print(f"200 records: {count(sample):,} Claude tokens vs {len(sample) // 4:,} by the chars/4 rule")

## 3. Packing by binary search
`pack` asks: what is the largest *n* such that the first *n* records fit in the target size?
Instead of trying every *n* (38,000 API calls), it halves the search range each step: about 16 calls per size.

In [ ]:
n_records = build_contexts(lines, list(SIZES.values()), count)
meta = {}
for name, size in SIZES.items():
    text = "\n".join(lines[:n_records[size]])
    (ROOT / f"data/processed/context_{name}.txt").write_text(text, encoding="utf-8")
    meta[name] = {"target_tokens": size, "n_records": n_records[size], "tokens": count(text)}
(ROOT / "data/processed/contexts.json").write_text(json.dumps(meta, indent=2))
# Only orders inside the biggest context are kept: these are the only reviews we pay to label.
records.iloc[:n_records[max(SIZES.values())]].to_csv(ROOT / "data/processed/records.csv", index=False)
pd.DataFrame(meta).T

Next: `uv run python scripts/label_reviews.py submit`, then `collect` when the batch has ended.